# GEN-KNOB Tuner — Qwen2.5-Coder LoRA Inference

This notebook uses the multi-adapter independent LoRA fine-tuning method over the `Qwen/Qwen2.5-Coder-7B-Instruct` base model. Unlike the previous MoE approach, this inference script dynamically loads either the PostgreSQL or MySQL specific standard LoRA adapter based on the targeted workload.

## Instructions
1. Run the installation and setup cells.
2. Ensure your `INPUT_FILE` points to your workload (e.g., job.json).
3. The script will automatically load the correct database LoRA adapter cleanly without task interference.

In [ ]:
# 1. Install dependencies
!pip install -q -U transformers datasets peft bitsandbytes accelerate huggingface_hub

In [ ]:
# 2. Imports and Initialization
import os, json, re, ast, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("Imports successful!")

In [ ]:
# 3. Advanced Prompt Preprocessing (Matching the Training Notebook)

def parse_hardware_spec(hw_raw: str) -> str:
    hw = str(hw_raw).lower()
    ram_match     = re.search(r"(\d+)\s*gb", hw)
    cores_match   = re.search(r"(\d+)\s*c(?:ore|pu)?(?:\b|[-_])", hw)
    threads_match = re.search(r"(\d+)\s*t(?:hread)?(?:\b|[-_])", hw)

    ram     = f"{ram_match.group(1)} GB"     if ram_match     else "unknown RAM"
    cores   = f"{cores_match.group(1)} cores"   if cores_match   else "unknown cores"
    threads = f"{threads_match.group(1)} threads" if threads_match else "unknown threads"

    return f"{ram} RAM, {cores}, {threads}"

def _extract_operator(node: dict) -> str:
    return node.get("Node Type", node.get("node_type", "Unknown"))

def _extract_cost(node: dict) -> float:
    return float(node.get("Total Cost", node.get("total_cost", node.get("cost", 0.0))))

def _flatten_plan_tree(node, operator_costs: dict):
    op   = _extract_operator(node)
    cost = _extract_cost(node)
    if op not in operator_costs:
        operator_costs[op] = [0.0, 0]
    operator_costs[op][0] += cost
    operator_costs[op][1] += 1
    for child_key in ("Plans", "plans", "children"):
        for child in node.get(child_key, []):
            _flatten_plan_tree(child, operator_costs)

def _parenthesise_str_plan(plan_str: str) -> str:
    pattern = re.compile(r"([A-Za-z][A-Za-z ]*?)\(cost=([\d.]+)\)")
    operator_costs: dict = {}
    for op, cost_str in pattern.findall(plan_str):
        op   = op.strip()
        cost = float(cost_str)
        if op not in operator_costs:
            operator_costs[op] = [0.0, 0]
        operator_costs[op][0] += cost
        operator_costs[op][1] += 1
    return _encode_operator_costs(operator_costs)

def _encode_operator_costs(operator_costs: dict) -> str:
    if not operator_costs:
        return "No plan"
    sorted_ops = sorted(
        operator_costs.items(),
        key=lambda kv: kv[1][0] / max(kv[1][1], 1),
        reverse=True
    )
    parts = []
    for op, (total, count) in sorted_ops:
        avg = total / max(count, 1)
        parts.append(f"{op}(cost={avg:.1f})")
    result = parts[0]
    for p in parts[1:]:
        result = f"{result}({p})"
    return result

def encode_query_plans(plans_raw) -> str:
    if not plans_raw:
        return "No query plans available."
    encoded = []
    for plan in plans_raw:
        if isinstance(plan, dict):
            op_costs: dict = {}
            _flatten_plan_tree(plan, op_costs)
            encoded.append(_encode_operator_costs(op_costs))
        elif isinstance(plan, str) and plan.strip():
            encoded.append(_parenthesise_str_plan(plan.strip()))
    return " | ".join(encoded) if encoded else "No query plans available."

def humanize_number(val) -> str:
    try:
        n = float(val)
    except (TypeError, ValueError):
        return str(val)
    abs_n = abs(n)
    sign  = "-" if n < 0 else ""
    if abs_n >= 1_000_000_000:
        return f"{sign}{abs_n / 1_000_000_000:.1f} billion"
    elif abs_n >= 1_000_000:
        return f"{sign}{abs_n / 1_000_000:.1f} million"
    elif abs_n >= 1_000:
        return f"{sign}{abs_n / 1_000:.1f} thousand"
    elif abs_n == 0:
        return "0"
    else:
        return f"{sign}{abs_n:.4g}"

def humanize_metrics(metrics: dict) -> dict:
    return {k: humanize_number(v) for k, v in metrics.items() if v != 0}

def format_qwen_inference_prompt(payload: dict, target_knobs: list = None) -> str:
    db_name   = str(payload.get('database', 'UNKNOWN')).upper()
    hw_parsed = parse_hardware_spec(payload.get('hardware_specs', ''))

    raw_metrics  = payload.get('internal_metrics', {}) or {}
    metrics_str  = ", ".join(f"{k} = {v}" for k, v in humanize_metrics(raw_metrics).items())

    features     = payload.get('workload_features', {}) or {}
    features_str = ", ".join(f"{k} = {v}" for k, v in features.items())

    q_plan_str = encode_query_plans(payload.get('query_plans', []))

    knobs_instruction = ""
    if target_knobs:
        knobs_str = ", ".join(target_knobs)
        knobs_instruction = f"IMPORTANT: You must ONLY provide configurations for the following {db_name} knobs:\n{knobs_str}\n\n"

    instruction = (
        f"You are an expert {db_name} Database Administrator.\n"
        f"The server hardware is: {hw_parsed}.\n\n"
        "Given the internal system metrics, workload characteristics, and query execution plans below, "
        "recommend the optimal discrete bucket configuration for each database knob.\n\n"
        f"{knobs_instruction}"
        f"INTERNAL SYSTEM METRICS:\n{metrics_str}\n\n"
        f"WORKLOAD FEATURES:\n{features_str}\n\n"
        f"QUERY PLANS:\n{q_plan_str}"
    )

    # Qwen/ChatML prompt construction (matches the target structure in the training notebook)
    prompt = f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n"
    return prompt

print("Prompt preprocessing tools initialized!")

In [ ]:
# 4. Input Target Payload
import json
import os

INPUT_FILE = "/kaggle/input/datasets/nisith210144g/job-bm/job.json"
TARGET_DATABASE = "mysql"  # Switch to 'postgresql' if needed
TARGET_HARDWARE = "hetzner-4c-8t-32gb"

# Fallback for local workspace testing if Kaggle path isn't found
if not os.path.exists(INPUT_FILE) and os.path.exists("../inferencing/mysql/job/job.json"):
    INPUT_FILE = "../inferencing/mysql/job/job.json"
elif not os.path.exists(INPUT_FILE):
    INPUT_FILE = "job.json" # Generic fallback

print(f"Loading inference payload from {INPUT_FILE}...")
with open(INPUT_FILE, 'r') as f:
    inference_payload = json.load(f)

# Load target knobs to constrain the model's output
target_knobs = []
knob_file = f"/home/E2ETune-AI4DB/knob_config/{TARGET_DATABASE}64_knob_config.json"
if not os.path.exists(knob_file):
    knob_file = f"../knob_config/{TARGET_DATABASE}64_knob_config.json"

if os.path.exists(knob_file):
    with open(knob_file, 'r') as f:
        knob_data = json.load(f)
        target_knobs = list(knob_data.keys())
    print(f"Loaded {len(target_knobs)} target knobs for {TARGET_DATABASE}.")
else:
    print(f"Warning: Knob config file for {TARGET_DATABASE} not found!")
    # Hardcoded fallback for mysql specifically to prevent Postgres outputs
    if TARGET_DATABASE == "mysql":
        target_knobs = [
            "max_connections", "innodb_log_buffer_size", "innodb_buffer_pool_size", "innodb_redo_log_capacity",
            "binlog_group_commit_sync_delay", "binlog_group_commit_sync_no_delay_count", "innodb_lock_wait_timeout",
            "innodb_stats_persistent_sample_pages", "innodb_read_io_threads", "innodb_write_io_threads",
            "optimizer_search_depth", "innodb_sort_buffer_size", "myisam_sort_buffer_size", "sort_buffer_size",
            "join_buffer_size", "tmp_table_size", "max_heap_table_size", "temptable_max_ram", "temptable_max_mmap",
            "innodb_page_cleaners", "innodb_lru_scan_depth", "innodb_max_dirty_pages_pct", "innodb_io_capacity",
            "innodb_io_capacity_max", "innodb_flush_log_at_timeout", "innodb_purge_threads", "innodb_purge_batch_size",
            "innodb_flush_neighbors"
        ]
        print(f"Using hardcoded fallback MySQL target knobs ({len(target_knobs)} knobs).")

# Inject database and hardware explicitly if missing
if "database" not in inference_payload:
    inference_payload["database"] = TARGET_DATABASE
if "hardware_specs" not in inference_payload:
    inference_payload["hardware_specs"] = TARGET_HARDWARE

formatted_prompt = format_qwen_inference_prompt(inference_payload, target_knobs)
print("── Formatted Inference Prompt ──")
print(formatted_prompt)

In [ ]:
# 5. Load Base Model and Standard LoRA Adapter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Dynamically select the correct adapter based on the targeted database to prevent task interference.
if TARGET_DATABASE == "mysql":
    ADAPTER_NAME = "NisithDissanayake/genknob-tuner-mysql-adapter"
else:
    ADAPTER_NAME = "NisithDissanayake/genknob-tuner-pg-adapter"

print(f"Loading Base Model ({BASE_MODEL_ID}) in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_cache=True
)
base_model.eval()

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading Standard LoRA Adapter ({ADAPTER_NAME})...")
# Because these are standard LoRA adapters (not MoE), we don't need any MixLoRA skeleton hacks here!
# We can cleanly wrap the base model directly.
model = PeftModel.from_pretrained(base_model, ADAPTER_NAME)
model.eval()

print("\nModel ready for inference!")

In [ ]:
# 6. Output Extraction Helper

def extract_json(s: str):
    m = re.search(r"\{.*?\}", s, flags=re.DOTALL)
    if not m: return None
    block = m.group(0)
    try: return json.loads(block)
    except Exception:
        try: return ast.literal_eval(block)
        except Exception: return None

In [ ]:
# 7. Generate Configurations
print("Generating Diverse LoRA Configurations...")
all_configs = []

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
prompt_length = inputs["input_ids"].shape[1]

for i in range(8):
    print(f"\n[{i+1}/8] Generating...")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024,
            temperature=0.7,
            do_sample=True,
            repetition_penalty=1.1,
            top_p=0.95,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Slice off the prompt to only decode the response
    response_tokens = outputs[0][prompt_length:]
    
    text = tokenizer.decode(response_tokens, skip_special_tokens=True)
    
    pred_dict = extract_json(text)
    
    if pred_dict:
        all_configs.append(pred_dict)
        print(f"✓ Extracted successfully: {len(pred_dict)} knobs generated.")
        print(json.dumps(pred_dict, indent=2))
    else:
        print("Failed to extract JSON. Raw response:\n", text[:200])

output_file = "/kaggle/working/qwen_generated_configs.json"
os.makedirs(os.path.dirname(output_file), exist_ok=True) 

with open(output_file, 'w') as f:
    json.dump(all_configs, f, indent=2)

print(f"\nDone! ✓ Saved {len(all_configs)} configurations to {output_file}")

In [ ]:
# 8. Bin Best Config (configuration keys only) using generated configs as reference
# NOTE: best_config.json has top-level metadata keys (best_cost, default_objective, etc.)
# We must extract ONLY the nested 'configuration' dict before binning — not the whole file.

import numpy as np

BEST_CONFIG_FILE = "best_config.json"  # Update path if needed

with open(BEST_CONFIG_FILE, 'r') as f:
    best_config_raw = json.load(f)

# --- FIX: extract only the 'configuration' sub-dict, ignore metadata fields ---
if "configuration" in best_config_raw:
    best_config = best_config_raw["configuration"]
else:
    # Fallback: assume the whole file is already a flat knob->value dict
    best_config = best_config_raw

print(f"Knobs to bin: {list(best_config.keys())}")

def make_bins_from_configs(configs: list, n_bins: int = 64) -> dict:
    """Derive per-knob bin edges from the generated candidate configs."""
    knob_values: dict = {}
    for cfg in configs:
        for k, v in cfg.items():
            try:
                knob_values.setdefault(k, []).append(float(v))
            except (TypeError, ValueError):
                pass
    bin_edges = {}
    for k, vals in knob_values.items():
        lo, hi = min(vals), max(vals)
        if lo == hi:
            # All generated values identical — widen slightly so np.linspace works
            lo, hi = lo * 0.9, hi * 1.1 if hi != 0 else 1.0
        bin_edges[k] = np.linspace(lo, hi, n_bins + 1)
    return bin_edges

def assign_bin(value: float, edges: np.ndarray) -> int:
    """Return 0-based bucket index for a raw value given bin edges."""
    idx = int(np.searchsorted(edges, value, side='right')) - 1
    return int(np.clip(idx, 0, len(edges) - 2))

# Build bin edges from generated configs (only knobs present in both)
if all_configs:
    bin_edges = make_bins_from_configs(all_configs, n_bins=64)
else:
    print("Warning: no generated configs found — cannot derive bin edges. Run cell 7 first.")
    bin_edges = {}

# Map each knob in best_config['configuration'] to its bucket index
binned_best_config = {}
skipped = []
for knob, raw_val in best_config.items():
    if knob not in bin_edges:
        skipped.append(knob)
        continue
    try:
        binned_best_config[knob] = assign_bin(float(raw_val), bin_edges[knob])
    except (TypeError, ValueError):
        skipped.append(knob)

if skipped:
    print(f"Skipped (no bin edges or non-numeric): {skipped}")

print("\nBinned best config (knob -> bucket index):")
print(json.dumps(binned_best_config, indent=2))

# Save
binned_output_file = "/kaggle/working/binned_best_config.json"
with open(binned_output_file, 'w') as f:
    json.dump(binned_best_config, f, indent=2)
print(f"\nSaved binned config to {binned_output_file}")
